In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import pickle
import copy

from imports import *
from config import dir_config, ephys_config
from src.utils import dpca_utils, dpca_plot_utils

In [ ]:
compiled_dir  = Path(dir_config.data.compiled)
processed_dir = Path(dir_config.data.processed)

## Load data

Data loading stays in the notebook. The utils start after you have
`session_metadata`, `neuron_metadata`, `ephys`, and the trial-info dicts.

In [ ]:
session_to_exclude = ["210210_GP_JP", "241209_GP_TZ"]

session_metadata = pd.read_csv(Path(processed_dir, "sessions_metadata.csv"))
session_metadata = session_metadata[~session_metadata.session_id.isin(session_to_exclude)].reset_index(drop=True)

neuron_metadata = pd.read_csv(Path(processed_dir, "neuron_metadata.csv"))
neuron_metadata = neuron_metadata[~neuron_metadata.session_id.isin(session_to_exclude)].reset_index(drop=True)

with open(Path(processed_dir, "glm_hmm_models", "glm_hmm_masked_final.pkl"), "rb") as f:
    glm_hmm = pickle.load(f)
glm_hmm_original = copy.deepcopy(glm_hmm)

with open(Path(processed_dir, "ephys_neuron_wise.pkl"), "rb") as f:
    ephys = pickle.load(f)

## Extract trial info (blocks or glm-hmm states)

In [ ]:
data = glm_hmm["data"]

# HMM states — also flips sign for awayRF sessions in-place on glm_hmm["data"]
# compiled_dir loads reaction_time from trial CSVs (required by get_trial_num)
biased_state_trial_info, unbiased_state_trial_info, state_occupancy = \
    dpca_utils.extract_hmm_state_trial_info(session_metadata, glm_hmm_original, data,
                                            compiled_dir=compiled_dir)

# Blocks from prob_toRF (run after extract_hmm_state_trial_info so awayRF sign flip is applied)
equal_block_trial_info, unequal_block_trial_info = \
    dpca_utils.extract_block_trial_info(data, session_metadata["session_id"])

## Shared setup

In [ ]:
toRF_sessions  = session_metadata.session_id[session_metadata.prior_direction == "toRF"]
awayRF_sessions = session_metadata.session_id[session_metadata.prior_direction == "awayRF"]

alignments = list(ephys_config["alignment_settings_GP"].keys())  # ['baseline','visual','cue','response']
marginalization_keys = ['b', 's', 'c', 't']

condition_dict_states = {
    "state_values": ["biased", "unbiased"],
    "coherences":   [0, 0.06, 0.2, 0.5],
    "choices":      ["awayRF", "toRF"],
}

state_trial_info = {
    "biased":   biased_state_trial_info,
    "unbiased": unbiased_state_trial_info,
}

COH_LABELS = ["0%", "6%", "20%", "50%"]


---
## Example 1 — All neurons, HMM states (baseline)

In [ ]:
_out = Path("../dissemination/dpca/dpca_toRF_session_all_neuron")
_out.mkdir(parents=True, exist_ok=True)


In [ ]:
neuron_ids = dpca_utils.get_neuron_ids(neuron_metadata, toRF_sessions)

avg, tw = dpca_utils.create_dpca_matrix(
    toRF_sessions, condition_dict_states, neuron_ids,
    state_trial_info, neuron_metadata, ephys, ephys_config,
    condition_type="states",
)
fit_avg, fit_tw, full_avg, full_tw = dpca_utils.clean_dpca_data(avg, tw, alignments)

dpca_results  = dpca_utils.fit_dpca_all_alignments(fit_avg, fit_tw, alignments, marginalization_keys=marginalization_keys)
projections   = dpca_utils.cross_period_projection(dpca_results, fit_avg, alignments)
time_axes     = dpca_utils.build_time_axes(fit_avg, ephys_config)


Plot variance explained, self-projection and cross-projection

In [ ]:
margs_to_plot = ['b', 's', 'c', 't']
n_components  = 3
fig = dpca_plot_utils.plot_variance_explained(dpca_results, alignments, margs_to_plot, n_components=n_components)
plt.show()


In [ ]:
_dpca_dir = Path(processed_dir) / 'dpca'
_dpca_dir.mkdir(parents=True, exist_ok=True)
_sig_path = _dpca_dir / "toRF_session_all_neuron_significance_masks.pkl"

if _sig_path.exists():
    with open(_sig_path, "rb") as f:
        significance_masks = pickle.load(f)
    print("Loaded significance_masks from cache.")
else:
    significance_masks = {}
    for alignment in alignments:
        dpca_model = dpca_results[alignment]["model"]
        print(f"Computing significance masks for {alignment} alignment...")
        significance_masks[alignment], _, _ = \
            dpca_utils.dpca_significance_analysis(
                copy.deepcopy(dpca_model), fit_avg[alignment], fit_tw[alignment],
                n_shuffles=100, n_splits=50, n_consecutive=1,
                keys=['b', 's', 'c'],
                key_groups={'s': [[0, 1], [2, 3]]},
                smooth_sigma=10,
            )
    with open(_sig_path, "wb") as f:
        pickle.dump(significance_masks, f)
    print(f"Saved → {_sig_path}")

In [ ]:
# Significant bin counts per alignment / marginalization (PC1)
for alignment in alignments:
    for key in ['b', 's', 'c']:
        n_sig = significance_masks[alignment][key][0].sum()
        print(f"  {alignment}/{key}: {n_sig} significant bins (PC1)")


In [ ]:
PC = 0
margs_to_plot = ['s', 'c', 'b']

fig = dpca_plot_utils.plot_self_projection(
    projections, time_axes, alignments, margs_to_plot,
    significance_masks=significance_masks, PC=PC,
    coh_labels=COH_LABELS,
)
plt.show()
fig.savefig(_out / "self_projection.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved → {_out / 'self_projection.png'}")

In [ ]:
# ── Self-projection with 95% CI band on the thick condition means (Example 1 only) ──
# The band is the analytic SEM of the condition-balanced marginal projection (the thick
# line), propagated from per-cell trial statistics in fit_tw — same trials the sig test
# uses. Bands pulling apart = the condition means separate = encoding onset for that
# variable. Notebook-local so it does NOT affect the other self-projection figures.
#
#   thick(v,t) = sum_n w_n * Xbar_n(v,t),   w_n = D[marg][:, PC]
#   var(thick) = sum_n w_n^2 * (1/K^2) * sum_cells( s2_cell / n_cell )
# See dpca_utils.dpca_significance_analysis for the matching condition-balanced collapse.

# cell axes of fit_tw after trial-averaging: 0=neuron 1=state(b) 2=coh(s) 3=choice(c) 4=time
# collapse the NON-key conditions for each marginalization (equal weight per condition):
_MARG_COLLAPSE = {'s': (1, 3), 'c': (1, 2), 'b': (2, 3)}


def thick_line_sem(trialX, w, collapse_axes):
    """Analytic SEM of the condition-balanced marginal projection (thick line).

    trialX        : (n_trial, n_neuron, n_state, n_coh, n_choice, n_time)  = fit_tw[align]
    w             : (n_neuron,) axis loading = dpca_results[align]['model'].D[marg][:, PC]
    collapse_axes : non-key condition axes to average over (cell space, see _MARG_COLLAPSE)
    returns sem   : (n_target_vals, n_time)   aligned with the thick-line order
    """
    cell_mean = np.nanmean(trialX, axis=0)                 # (neuron, b, s, c, time)
    cell_var  = np.nanvar(trialX,  axis=0, ddof=1)         # per-cell trial variance
    cell_cnt  = np.sum(~np.isnan(trialX), axis=0)          # per-cell trial count
    with np.errstate(invalid='ignore', divide='ignore'):
        cell_sem2 = cell_var / cell_cnt                    # SEM^2 of each cell mean
    K = np.sum(~np.isnan(cell_mean), axis=collapse_axes, keepdims=True)  # #valid cells
    with np.errstate(invalid='ignore', divide='ignore'):
        var_Xbar = np.nansum(cell_sem2, axis=collapse_axes, keepdims=True) / K**2
    var_Xbar = np.squeeze(var_Xbar, axis=collapse_axes)    # (neuron, target, time)
    var_thick = np.tensordot(w**2, var_Xbar, axes=([0], [0]))  # (target, time)
    return np.sqrt(var_thick)


def plot_self_projection_ci(projections, time_axes, alignments, margs_to_plot,
                            fit_tw, dpca_results, significance_masks=None, PC=0,
                            n_states=2, n_coh=4, coh_labels=None, title=None,
                            trace_alpha=0.15, z=1.96, show_thin=True):
    """plot_self_projection + a z*SEM shaded band on each thick condition mean.
    Mirrors dpca_plot_utils.plot_self_projection; only the bands are added."""
    dpu = dpca_plot_utils
    biased_colors, unbiased_colors = dpu.BIASED_COLORS[:n_coh], dpu.UNBIASED_COLORS[:n_coh]
    if coh_labels is None:
        coh_labels = [f"coh{i}" for i in range(n_coh)]
    n_rows, n_cols = len(margs_to_plot), len(alignments)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows), squeeze=False)

    for col, alignment in enumerate(alignments):
        Z, time = projections[alignment][alignment], time_axes[alignment]
        D_model, trialX = dpca_results[alignment]["model"].D, fit_tw[alignment]
        for row, marg in enumerate(margs_to_plot):
            ax = axes[row][col]
            data_pc = Z[marg][PC]                          # (n_states, n_coh, n_choices, n_time)
            if show_thin:
                dpu.plot_dpca_traces(ax, time, data_pc, n_states, n_coh,
                                     biased_colors, unbiased_colors, alpha=trace_alpha)
            sem = thick_line_sem(trialX, D_model[marg][:, PC], _MARG_COLLAPSE[marg])  # (target,time)

            if marg == 's':
                mean = np.nanmean(data_pc, axis=(0, 2))    # (n_coh, time)
                colors, styles = [dpu._STIM_GREYS[s] for s in range(n_coh)], ["-"] * n_coh
            elif marg == 'c':
                mean = np.nanmean(data_pc, axis=(0, 1))    # (n_choices, time)  0=awayRF,1=toRF
                colors, styles = ["black", "black"], ["--", "-"]
            else:  # 'b'
                mean = np.nanmean(data_pc, axis=(1, 2))    # (n_states, time)
                colors, styles = [biased_colors[-1], unbiased_colors[-1]], ["-"] * n_states
            for k in range(mean.shape[0]):
                ax.plot(time, mean[k], color=colors[k], linestyle=styles[k], linewidth=2.5)
                ax.fill_between(time, mean[k] - z * sem[k], mean[k] + z * sem[k],
                                color=colors[k], alpha=0.22, linewidth=0)

            if significance_masks is not None and marg in significance_masks.get(alignment, {}):
                dpu.bar_significance(ax, time, significance_masks[alignment][marg][PC])
            if not show_thin:
                ax.axvline(0, color="k", linestyle="--", linewidth=0.8)
                ax.spines[["top", "right"]].set_visible(False)
            ax.set_yticks([])
            if row == 0:
                ax.set_title(dpu.ALIGN_LABELS.get(alignment, alignment), fontsize=13)
            if col == 0:
                ax.set_ylabel(f"{dpu.MARG_LABELS.get(marg, marg)} PC{PC + 1}", fontsize=12)
            if row == n_rows - 1:
                ax.set_xlabel("Time aligned to event (ms)", fontsize=11)

    if title:
        fig.suptitle(title, y=1.01)
    plt.tight_layout()
    return fig

In [ ]:
PC = 0
fig = plot_self_projection_ci(
    projections, time_axes, alignments, ['s', 'c', 'b'],
    fit_tw, dpca_results,
    significance_masks=significance_masks, PC=PC, coh_labels=COH_LABELS,
    title="Self-projection + 95% CI (analytic SEM) — all neurons (Example 1)",
)
plt.show()
# save into the processed folder (_dpca_dir = processed_dir/'dpca'), not dissemination
_ci_out = _dpca_dir / "self_projection_ci.png"
fig.savefig(_ci_out, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved → {_ci_out}")

In [ ]:
fig = dpca_plot_utils.plot_variance_explained(dpca_results, alignments, ['b', 's', 'c', 't'], n_components=3)
fig.savefig(_out / "variance_explained.png", dpi=150, bbox_inches="tight")
plt.close(fig)

In [ ]:
# Cross-period significance — rigorous shared-trial refit.
# Axes are refit on 80% of the FIT-epoch trials each split (once per split, shared across
# all proj epochs) and the held-out 20% are decoded in each PROJ epoch (no trial leakage).
# New cache key so stale refit=False masks aren't silently reused.
_cross_sig_path = _dpca_dir / "toRF_session_all_neuron_cross_sig_masks_refit.pkl"

if _cross_sig_path.exists():
    with open(_cross_sig_path, "rb") as f:
        cross_sig_masks = pickle.load(f)
    print("Loaded cross_sig_masks from cache.")
else:
    proj_Xs  = {pa: fit_avg[pa] for pa in alignments}
    proj_tws = {pa: fit_tw[pa]  for pa in alignments}
    cross_sig_masks = {}
    for fit_align in ["baseline", "response"]:
        dpca_model = dpca_results[fit_align]["model"]
        print(f"  cross sig: fit={fit_align} → all proj epochs...")
        # Returns masks[proj_align][proj_key][class_key].
        cross_sig_masks[fit_align], _, _ = dpca_utils.dpca_cross_significance_analysis(
            copy.deepcopy(dpca_model),
            fit_avg[fit_align], fit_tw[fit_align],   # FIT epoch  → refit axes on 80%
            proj_Xs, proj_tws,                       # all PROJ epochs decoded together
            n_shuffles=100, n_splits=15, n_consecutive=1,
            keys=['b', 'c'],
            smooth_sigma=10,
            cross_decode_keys={'b': ['b', 'c'], 'c': ['b', 'c']},
            seed=0,
        )
    with open(_cross_sig_path, "wb") as f:
        pickle.dump(cross_sig_masks, f)
    print(f"Saved → {_cross_sig_path}")

In [ ]:
fit_align = "baseline"
margs_to_plot_cp = ['b', 'c']
fig = dpca_plot_utils.plot_cross_projection(
    projections, time_axes, alignments, margs_to_plot_cp,
    fit_align, coh_labels=COH_LABELS,
    significance_masks=cross_sig_masks["baseline"],
)
plt.show()
fig.savefig(_out / "cross_projection_baseline.png", dpi=150, bbox_inches="tight")
plt.close(fig)

In [ ]:
fit_align = "response"
margs_to_plot_cp = ['b', 'c']
fig = dpca_plot_utils.plot_cross_projection(
    projections, time_axes, alignments, margs_to_plot_cp,
    fit_align, coh_labels=COH_LABELS,
    significance_masks=None,#cross_sig_masks["response"],
)
plt.show()
fig.savefig(_out / "cross_projection_response.png", dpi=150, bbox_inches="tight")
plt.close(fig)

### plot loadings

In [ ]:
from matplotlib.lines import Line2D

full_time_axes = dpca_utils.build_time_axes(full_avg, ephys_config)

COH_COLORS = ["#e31a1c", "#ff7f00", "#33a02c", "#1f78b4"]
TOP_N  = 15
N_COLS = 5
N_ROWS = TOP_N // N_COLS

SKIP_MARG = {"baseline": ['s'], "visual": ['s'], "cue": [], "response": []}

def _plot_psth_grid(top_idx, top_nids, top_w, time, psth_fn, title, legend_handles):
    """psth_fn(rank, nidx) → plots lines onto ax."""
    fig, axes = plt.subplots(N_ROWS, N_COLS,
                             figsize=(4 * N_COLS, 3 * N_ROWS),
                             sharex=True)
    for rank, nidx in enumerate(top_idx):
        ax = axes.flatten()[rank]
        psth_fn(ax, rank, nidx)
        ax.axvline(0, color='k', ls='--', lw=0.8)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.set_title(f"#{rank+1}  {top_nids[rank]}\nw={top_w[rank]:.3f}", fontsize=7)
        ax.tick_params(labelsize=7)
        ax.set_ylabel("FR (Hz)", fontsize=7)
    fig.legend(handles=legend_handles, bbox_to_anchor=(1.01, 0.98),
               loc='upper left', fontsize=9)
    fig.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()

for alignment in alignments:
    dpca_model = dpca_results_no_undefined[alignment]["model"]
    time       = full_time_axes[alignment]
    data       = full_avg[alignment]  # (n_neurons, n_states, n_coh, n_choices, n_time)
    # states: [0]=biased, [1]=unbiased  |  choices: [0]=awayRF, [1]=toRF

    for marg in ['b', 's', 'c']:
        if marg in SKIP_MARG[alignment]:
            continue

        decoder    = dpca_model.D[marg][:, 0]
        top_idx    = np.argsort(np.abs(decoder))[::-1][:TOP_N]
        top_nids   = neuron_ids_no_undefined[top_idx]
        top_w      = decoder[top_idx]
        marg_label = dpca_plot_utils.MARG_LABELS[marg]

        # ── neuron IDs ────────────────────────────────────────────────────────
        print(f"\n{alignment} / {marg_label} PC1 — top {TOP_N} neuron IDs:")
        print(list(top_nids))

        # ── loading bar chart ─────────────────────────────────────────────────
        fig_l, ax_l = plt.subplots(figsize=(10, 3))
        bar_colors = ['#d62728' if w >= 0 else '#1f77b4' for w in top_w]
        ax_l.bar(range(TOP_N), top_w, color=bar_colors)
        ax_l.set_xticks(range(TOP_N))
        ax_l.set_xticklabels(top_nids, rotation=90, fontsize=7)
        ax_l.axhline(0, color='k', lw=0.8)
        ax_l.set_ylabel("PC1 loading")
        ax_l.set_xlabel("neuron id")
        ax_l.spines['top'].set_visible(False)
        ax_l.spines['right'].set_visible(False)
        ax_l.set_title(f"Loadings — {alignment} / {marg_label} PC1  (ranked by |loading|)")
        plt.tight_layout()
        plt.show()

        # ── PSTH ─────────────────────────────────────────────────────────────
        if marg == 'b':
            # biased + unbiased in same panel, using dPCA self-projection color scheme
            bc = dpca_plot_utils.BIASED_COLORS
            uc = dpca_plot_utils.UNBIASED_COLORS

            def _draw_bias(ax, rank, nidx):
                for ci in range(4):
                    ax.plot(time, data[nidx, 0, ci, 1, :], color=bc[ci], lw=1.5, ls='-')   # biased toRF
                    ax.plot(time, data[nidx, 0, ci, 0, :], color=bc[ci], lw=1.0, ls='--')  # biased awayRF
                    ax.plot(time, data[nidx, 1, ci, 1, :], color=uc[ci], lw=1.5, ls='-')   # unbiased toRF
                    ax.plot(time, data[nidx, 1, ci, 0, :], color=uc[ci], lw=1.0, ls='--')  # unbiased awayRF

            legend_handles = (
                [Line2D([], [], color=bc[i], lw=1.5, label=f"{COH_LABELS[i]} biased")   for i in range(4)] +
                [Line2D([], [], color=uc[i], lw=1.5, label=f"{COH_LABELS[i]} unbiased") for i in range(4)] +
                [Line2D([], [], color='k', lw=1.5, ls='-',  label='toRF'),
                 Line2D([], [], color='k', lw=1.0, ls='--', label='awayRF')]
            )
            _plot_psth_grid(top_idx, top_nids, top_w, time, _draw_bias,
                            f"Top {TOP_N} neurons PSTH — {alignment} / Bias PC1  (ranked by |loading|)",
                            legend_handles)

        else:
            # stimulus / choice: average over states, coherence colors
            def _draw_coh(ax, rank, nidx):
                psth = np.nanmean(data[nidx], axis=0)  # (n_coh, n_choices, n_time)
                for ci, color in enumerate(COH_COLORS):
                    ax.plot(time, psth[ci, 1, :], color=color, lw=1.5, ls='-')
                    ax.plot(time, psth[ci, 0, :], color=color, lw=1.0, ls='--')

            legend_handles = (
                [Line2D([], [], color=COH_COLORS[i], lw=1.5, label=COH_LABELS[i]) for i in range(4)] +
                [Line2D([], [], color='k', lw=1.5, ls='-',  label='toRF'),
                 Line2D([], [], color='k', lw=1.0, ls='--', label='awayRF')]
            )
            _plot_psth_grid(top_idx, top_nids, top_w, time, _draw_coh,
                            f"Top {TOP_N} neurons PSTH — {alignment} / {marg_label} PC1  (ranked by |loading|)",
                            legend_handles)


---
## Cell-type leave-out — self-projection

Exclude each cell type in turn (plus visuomotor+motor combined); fit dPCA and plot self-projection.
Results saved to `dissemination/dpca/dpca_toRF_cell_type_leaveout/<tag>/`.

In [ ]:
_out_ct = Path("../dissemination/dpca/dpca_toRF_cell_type_leaveout")
_out_ct.mkdir(parents=True, exist_ok=True)

cell_type_configs = [
    ("no_undefined",        ["trash", "undefined"]),
    ("no_visuomotor",       ["trash", "visuomotor"]),
    ("no_motor",            ["trash", "motor"]),
    ("no_visual_phasic",    ["trash", "visual_phasic"]),
    ("no_visual_tonic",     ["trash", "visual_tonic"]),
    ("no_visuomotor_motor", ["trash", "visuomotor", "motor"]),
]

for tag, exclude_types in cell_type_configs:
    print(f"\n{'='*60}\n  Exclude: {exclude_types}\n{'='*60}")
    out_dir = _out_ct / tag
    out_dir.mkdir(parents=True, exist_ok=True)
    _sig_path_ct = _dpca_dir / f"toRF_cell_type_{tag}_significance_masks.pkl"

    nids = dpca_utils.get_neuron_ids(neuron_metadata, toRF_sessions, exclude_cell_types=exclude_types)
    print(f"  Neurons: {len(nids)}")

    avg_ct, tw_ct = dpca_utils.create_dpca_matrix(
        toRF_sessions, condition_dict_states, nids,
        state_trial_info, neuron_metadata, ephys, ephys_config,
        condition_type="states",
    )
    fa_ct, ftw_ct, _, _ = dpca_utils.clean_dpca_data(avg_ct, tw_ct, alignments)
    results_ct = dpca_utils.fit_dpca_all_alignments(fa_ct, ftw_ct, alignments, marginalization_keys=marginalization_keys)
    proj_ct = dpca_utils.cross_period_projection(results_ct, fa_ct, alignments)
    ta_ct = dpca_utils.build_time_axes(fa_ct, ephys_config)

    if _sig_path_ct.exists():
        with open(_sig_path_ct, "rb") as f:
            sig_ct = pickle.load(f)
        print(f"  Loaded sig from cache.")
    else:
        sig_ct = {}
        for alignment in alignments:
            print(f"    sig: {alignment}...")
            sig_ct[alignment], _, _ = dpca_utils.dpca_significance_analysis(
                copy.deepcopy(results_ct[alignment]["model"]), fa_ct[alignment], ftw_ct[alignment],
                n_shuffles=100, n_splits=50, n_consecutive=1,
                keys=['b', 's', 'c'], key_groups={'s': [[0, 1], [2, 3]]},
                smooth_sigma=10,
            )
        with open(_sig_path_ct, "wb") as f:
            pickle.dump(sig_ct, f)
        print(f"  Saved → {_sig_path_ct}")

    fig = dpca_plot_utils.plot_self_projection(
        proj_ct, ta_ct, alignments, ['s', 'c', 'b'],
        significance_masks=sig_ct, PC=0, coh_labels=COH_LABELS,
        title=f"Self-projection — exclude {exclude_types}",
    )
    plt.show()
    fig.savefig(out_dir / "self_projection.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {out_dir / 'self_projection.png'}")

---
## awayRF sessions — all neurons

Full dPCA analysis (self-projection + cross-projection) on awayRF prior sessions.
Results saved to `dissemination/dpca/dpca_awayRF_session_all_neuron/`.

In [ ]:
_out_away = Path("../dissemination/dpca/dpca_awayRF_session_all_neuron")
_out_away.mkdir(parents=True, exist_ok=True)
_sig_away_path       = _dpca_dir / "awayRF_session_all_neuron_significance_masks.pkl"
_cross_sig_away_path = _dpca_dir / "awayRF_session_all_neuron_cross_sig_masks_refit.pkl"

neuron_ids_away = dpca_utils.get_neuron_ids(neuron_metadata, awayRF_sessions)
print(f"awayRF neurons: {len(neuron_ids_away)}")

avg_away, tw_away = dpca_utils.create_dpca_matrix(
    awayRF_sessions, condition_dict_states, neuron_ids_away,
    state_trial_info, neuron_metadata, ephys, ephys_config,
    condition_type="states",
)
fit_avg_away, fit_tw_away, _, _ = dpca_utils.clean_dpca_data(avg_away, tw_away, alignments)

dpca_results_away = dpca_utils.fit_dpca_all_alignments(
    fit_avg_away, fit_tw_away, alignments, marginalization_keys=marginalization_keys
)
projections_away = dpca_utils.cross_period_projection(dpca_results_away, fit_avg_away, alignments)
time_axes_away = dpca_utils.build_time_axes(fit_avg_away, ephys_config)

if _sig_away_path.exists():
    with open(_sig_away_path, "rb") as f:
        sig_away = pickle.load(f)
    print("Loaded sig_away from cache.")
else:
    sig_away = {}
    for alignment in alignments:
        print(f"  sig: {alignment}...")
        sig_away[alignment], _, _ = dpca_utils.dpca_significance_analysis(
            copy.deepcopy(dpca_results_away[alignment]["model"]),
            fit_avg_away[alignment], fit_tw_away[alignment],
            n_shuffles=100, n_splits=50, n_consecutive=1,
            keys=['b', 's', 'c'], key_groups={'s': [[0, 1], [2, 3]]},
            smooth_sigma=10,
        )
    with open(_sig_away_path, "wb") as f:
        pickle.dump(sig_away, f)
    print(f"Saved → {_sig_away_path}")

fig = dpca_plot_utils.plot_self_projection(
    projections_away, time_axes_away, alignments, ['s', 'c', 'b'],
    significance_masks=sig_away, PC=0, coh_labels=COH_LABELS,
    title="Self-projection — awayRF sessions",
)
plt.show()
fig.savefig(_out_away / "self_projection.png", dpi=150, bbox_inches="tight")
plt.close(fig)

# Cross-period significance — rigorous shared-trial refit (see Example 1 for details).
if _cross_sig_away_path.exists():
    with open(_cross_sig_away_path, "rb") as f:
        cross_sig_away = pickle.load(f)
    print("Loaded cross_sig_away from cache.")
else:
    proj_Xs_away  = {pa: fit_avg_away[pa] for pa in alignments}
    proj_tws_away = {pa: fit_tw_away[pa]  for pa in alignments}
    cross_sig_away = {}
    for fit_align in ["baseline", "response"]:
        print(f"  cross sig: fit={fit_align} → all proj epochs...")
        cross_sig_away[fit_align], _, _ = dpca_utils.dpca_cross_significance_analysis(
            copy.deepcopy(dpca_results_away[fit_align]["model"]),
            fit_avg_away[fit_align], fit_tw_away[fit_align],      # FIT epoch → refit axes on 80%
            proj_Xs_away, proj_tws_away,                          # all PROJ epochs decoded together
            n_shuffles=100, n_splits=15, n_consecutive=1,
            keys=['b', 'c'], smooth_sigma=10,
            cross_decode_keys={'b': ['b', 'c'], 'c': ['b', 'c']},
            seed=0,
        )
    with open(_cross_sig_away_path, "wb") as f:
        pickle.dump(cross_sig_away, f)
    print(f"Saved → {_cross_sig_away_path}")

for fit_align in ["baseline", "response"]:
    fig = dpca_plot_utils.plot_cross_projection(
        projections_away, time_axes_away, alignments, ['b', 'c'],
        fit_align, coh_labels=COH_LABELS,
        significance_masks=cross_sig_away[fit_align],
    )
    plt.show()
    fig.savefig(_out_away / f"cross_projection_{fit_align}.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {_out_away / f'cross_projection_{fit_align}.png'}")

---
## Example 5 — Blocks instead of HMM states

Block membership comes from `glm_hmm["data"][session_id]["prob_toRF"]`:
- `prob_toRF == 50` → equal block
- `prob_toRF != 50` → unequal block

In [ ]:
condition_dict_blocks = {
    "state_values": ["equal", "unequal"],
    "coherences":   [0, 0.06, 0.2, 0.5],
    "choices":      ["awayRF", "toRF"],
}

block_trial_info = {
    "equal":   equal_block_trial_info,
    "unequal": unequal_block_trial_info,
}

neuron_ids = dpca_utils.get_neuron_ids(neuron_metadata, toRF_sessions)

avg, tw = dpca_utils.create_dpca_matrix(
    toRF_sessions, condition_dict_blocks, neuron_ids,
    block_trial_info, neuron_metadata, ephys, ephys_config,
    condition_type="blocks",
)
fit_avg, fit_tw, full_avg, full_tw = dpca_utils.clean_dpca_data(avg, tw, alignments)

dpca_results_blocks = dpca_utils.fit_dpca_all_alignments(fit_avg, fit_tw, alignments)
projections_blocks  = dpca_utils.cross_period_projection(dpca_results_blocks, full_avg, alignments)

---
## Example 6 — Combining filters: exclude trash, 10% leave-out, blocks

In [ ]:
neuron_ids_combined = dpca_utils.get_neuron_ids(
    neuron_metadata, toRF_sessions,
    exclude_cell_types=["trash"],
    leave_out_fraction=0.1,
    rng=np.random.default_rng(0),
)

avg, tw = dpca_utils.create_dpca_matrix(
    toRF_sessions, condition_dict_blocks, neuron_ids_combined,
    block_trial_info, neuron_metadata, ephys, ephys_config,
    condition_type="blocks",
)
fit_avg, fit_tw, full_avg, full_tw = dpca_utils.clean_dpca_data(avg, tw, alignments)

dpca_results_combined = dpca_utils.fit_dpca_all_alignments(fit_avg, fit_tw, alignments)
projections_combined  = dpca_utils.cross_period_projection(dpca_results_combined, full_avg, alignments)